# 03 - Embedded-cluster CASSCF/NEVPT2: Ce3+:LiYF4 (4f-5d) and Ti3+:Al2O3 (3d-3d)

Tier `gainmat.cluster`. Each centre is treated as dopant + first anion shell in QM, the next
cation shell as ECP-only pseudo-atoms where a full-core ECP exists (Li+, Al3+), and a ~20 A sphere of
point charges whose boundary charges are fitted so the potential matches the Ewald potential of the
infinite crystal (via `pyscf.qmmm.mm_charge`).

* **Ti3+:Al2O3**: ROHF -> SA-CASSCF(1,5) -> SC-NEVPT2, vertical 2T2g -> 2Eg.
* **Ce3+:LiYF4**: ROHF -> SA-CASSCF(1,12) (4f+5d) -> SC-NEVPT2 -> 12x12 ab initio ligand-field matrix + spin-orbit
  (zeta_4f = 615, zeta_5d = 1082 cm-1).

Only vertical energies at the ground-state geometry are computed: there is no excited-state relaxation, so
**no Stokes shift**, and the Ti Jahn-Teller shoulder is not predicted. Expect errors of a few thousand cm-1.

Runtime and engine: PySCF on CPU. gpu4pyscf is not used, because its CASSCF/NEVPT2, ROHF + QM/MM point
charges and ECP-cap support is incomplete. Select a high-RAM runtime: the CPU core count is what matters,
and the A100 sits idle for this tier. Expected runtimes for the `full` level: Ti about 0.5-2 h, Ce with
ANO-RCC about 2-6 h. For a quicker Ce run, set `CE_FULL_BASIS=x2c-tzvpall`.

In [ ]:
# Install. Note: pyscf already includes mcscf, mrpt (NEVPT2) and qmmm
!pip -q install pyscf ase nbformat matplotlib

In [ ]:
# Upload gainmat.zip (the repo folder) to /content, or mount Drive and copy it there
import os, sys, zipfile, glob
if not os.path.isdir('/content/gainmat') and os.path.exists('/content/gainmat.zip'):
    zipfile.ZipFile('/content/gainmat.zip').extractall('/content')
# accept either /content/gainmat/gainmat (repo) or /content/gainmat (package)
root = '/content/gainmat' if os.path.isdir('/content/gainmat/gainmat') else '/content'
sys.path.insert(0, root)
import gainmat; print('gainmat from', gainmat.__file__)

In [ ]:
# Threads: set before PySCF does heavy work
import os, multiprocessing
os.environ['OMP_NUM_THREADS'] = str(multiprocessing.cpu_count())
from pyscf import lib
lib.num_threads(multiprocessing.cpu_count())
print('threads', lib.num_threads())
LEVEL = 'full'      # 'fast' for a smoke test (~5-10 min total)
# os.environ['CE_FULL_BASIS'] = 'x2c-tzvpall'   # cheaper all-electron Ce basis

In [ ]:
# Sanity: embedding geometry + Madelung potential vs Ewald
from gainmat.cluster import embedding as emb
for args in [('Al2O3','Al','Ti',3,2.2,3.6), ('LiYF4','Y','Ce',3,2.6,4.0)]:
    cl = emb.make_cluster(*args[:4], r_qm=args[4], r_cap=args[5], r_pc=20.0 if LEVEL=='full' else 12.0)
    s = cl.summary(); print({k: s[k] for k in ['name','n_qm','n_cap','n_pc','dopant_ligand_A','total_charge','madelung_ewald_V','madelung_cluster_V','fit_rms_V','notes']})

In [ ]:
import time, json
from dataclasses import asdict
from gainmat.cluster.run_cluster import run_center, print_table
results, raw = {}, {}
for name in ['Ti:Al2O3', 'Ce:LiYF4']:
    t = time.time(); r = {}
    dp = run_center(name, LEVEL, raw_out=r)
    results[name] = asdict(dp); raw[name] = r
    print_table(name, dp); print(f'  wall {time.time()-t:.0f} s')

In [ ]:
# Computed-vs-experiment table (pandas)
import pandas as pd
from gainmat.cluster.run_cluster import compare_table
from gainmat.schema import DopantProps
rows = []
for name, d in results.items():
    dp = DopantProps(**d)
    for q, c, e, u in compare_table(name, dp):
        rows.append(dict(center=name, quantity=q, calc_cm1=c, exp_cm1=e,
                         calc_nm=(1e7/c if c else None), exp_nm=1e7/e, diff_cm1=(c-e if c else None)))
df = pd.DataFrame(rows); df.round(1)

In [ ]:
# Stick spectra: computed (up) vs experiment (down)
import matplotlib.pyplot as plt
from gainmat.refs import cluster_refs as R
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
ce = results['Ce:LiYF4']; ti = results['Ti:Al2O3']
ce_em = [ce['em_peak_nm']] if ce['em_peak_nm'] else []
ax = axs[0]
ax.vlines([1e7/x for x in R.CE_LIYF4['abs_5d_cm1']], 0, -1, color='k', label='exp. 4f-5d')
ax.vlines(ce['abs_peaks_nm'], 0, 1, color='C0', label='calc. NEVPT2+SO')
ax.vlines(R.CE_LIYF4['em_nm'], 0, -0.5, color='C3', label='exp. emission')
ax.vlines(ce_em, 0, 0.5, color='C1', label='calc. vertical emission')
ax.axhline(0, color='grey', lw=.5); ax.set_xlabel('wavelength (nm)'); ax.set_title('Ce3+:LiYF4'); ax.legend(fontsize=7); ax.set_yticks([])
ax = axs[1]
ax.vlines(R.TI_AL2O3['abs_nm'], 0, -1, color='k', label='exp. abs (main, JT shoulder)')
ax.vlines([R.TI_AL2O3['em_nm']], 0, -0.5, color='C3', label='exp. emission')
ax.vlines(ti['abs_peaks_nm'], 0, 1, color='C0', label='calc. 2T2g->2Eg')
ax.axhline(0, color='grey', lw=.5); ax.set_xlabel('wavelength (nm)'); ax.set_title('Ti3+:Al2O3'); ax.legend(fontsize=7); ax.set_yticks([])
plt.tight_layout(); os.makedirs('/content/results', exist_ok=True); plt.savefig('/content/results/cluster_sticks.png', dpi=150); plt.show()

In [ ]:
os.makedirs('/content/results', exist_ok=True)
with open('/content/results/cluster_results.json', 'w') as f:
    json.dump(dict(level=LEVEL, results=results, raw=raw), f, indent=1, default=float)
print('saved /content/results/cluster_results.json')